<a href="https://colab.research.google.com/github/aarushi-202618017/202618017_Aarushi_DS605/blob/main/202618017_LAB_05/202618017_LAB_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part A - Scikit-learn Implementation**

In [ ]:
!pip install ucimlrepo -q

In [ ]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


In [ ]:
garment_repo = fetch_ucirepo(id=597)
X_raw = garment_repo.data.features.copy()
y_raw = garment_repo.data.targets.copy()

In [ ]:
df_x= pd.DataFrame(X_raw)
df_y= pd.DataFrame(y_raw)

In [ ]:
df= pd.concat([df_x,df_y],axis=1)

In [ ]:
df.head()

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: f

In [ ]:
df.describe()

,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
count,1197.000000,1197.000000,1197.000000,691.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000
mean,6.426901,0.729632,15.062172,1190.465991,4567.460317,38.210526,0.730159,0.369256,0.150376,34.609858,0.735091
std,3.463963,0.097891,10.943219,1837.455001,3348.823563,160.182643,12.709757,3.268987,0.427848,22.197687,0.174488
min,1.000000,0.070000,2.900000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.233705
25%,3.000000,0.700000,3.940000,774.500000,1440.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.650307
50%,6.000000,0.750000,15.260000,1039.000000,3960.000000,0.000000,0.000000,0.000000,0.000000,34.000000,0.773333
75%,9.000000,0.800000,24.260000,1252.500000,6960.000000,50.000000,0.000000,0.000000,0.000000,57.000000,0.850253
max,12.000000,0.800000,54.560000,23122.000000,25920.000000,3600.000000,300.000000,45.000000,2.000000,89.000000,1.120437


In [ ]:
df['department'].unique()

array(['sweing', 'finishing'], dtype=object)

In [ ]:
(df['department'] == 'finishing ').sum()

np.int64(0)

In [ ]:
X = df.drop(columns=['date', 'actual_productivity'], errors='ignore')

In [ ]:
categorical_cols = ['quarter', 'department', 'day']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

In [ ]:
y_reg = df['actual_productivity'].values
y_cls = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

In [ ]:
X_train, X_test, y_train_reg, y_test_reg, y_train_cls, y_test_cls = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42)

Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_transform= Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transform= Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))])

col_transform = ColumnTransformer(
    transformers=[
        ('num', num_transform, numeric_cols),
        ('cat', cat_transform, categorical_cols)
    ]
)

In [ ]:
X_train_prep= col_transform.fit_transform(X_train)
X_test_prep= col_transform.transform(X_test)

In [ ]:
import time

t0= time.perf_counter()
model1 = LinearRegression()
model1.fit(X_train_prep, y_train_reg)
lr_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_reg_sk = model1.predict(X_test_prep)
lr_pred_time = time.perf_counter() - t0

mae_sk = mean_absolute_error(y_test_reg, y_pred_reg_sk)
rmse_sk = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_sk))
r2_sk = r2_score(y_test_reg, y_pred_reg_sk)

# logistic model
t0= time.perf_counter()
model2= LogisticRegression()
model2.fit(X_train_prep, y_train_cls)
logr_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_cls_sk = model2.predict(X_test_prep)
logr_pred_time = time.perf_counter() - t0

acc_sk = accuracy_score(y_test_cls, y_pred_cls_sk)
prec_sk = precision_score(y_test_cls, y_pred_cls_sk)
rec_sk = recall_score(y_test_cls, y_pred_cls_sk)
f1_sk = f1_score(y_test_cls, y_pred_cls_sk)

In [ ]:
print("Linear Regression:")
print(f"MAE:              {mae_sk:.4f}")
print(f"RMSE:             {rmse_sk:.4f}")
print(f"R² Score:         {r2_sk:.4f}")
print(f"Training Time:    {lr_time * 1000:.3f} ms")
print(f"Prediction Time:  {lr_pred_time * 1000:.3f} ms\n")

print("Logistic Regression:")
print(f"Accuracy:         {acc_sk:.4f}")
print(f"Precision:        {prec_sk:.4f}")
print(f"Recall:           {rec_sk:.4f}")
print(f"F1-Score:         {f1_sk:.4f}")
print(f"Training Time:    {logr_time * 1000:.3f} ms")
print(f"Prediction Time:  {logr_pred_time * 1000:.3f} ms")

Linear Regression:
MAE:              0.1088
RMSE:             0.1481
R² Score:         0.1736
Training Time:    24.441 ms
Prediction Time:  0.701 ms

Logistic Regression:
Accuracy:         0.7417
Precision:        0.7626
Recall:           0.9435
F1-Score:         0.8434
Training Time:    16.331 ms
Prediction Time:  0.649 ms


# **Part B - From-Scratch Implementation**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
Xs_train= X_train
Xs_test = X_test
ys_cls= y_train_cls
ys_reg= y_train_reg
ys_test_cls= y_test_cls
ys_test_reg= y_test_reg

In [34]:
tnum_cols = Xs_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
tcat_cols = Xs_train.select_dtypes(include=['object']).columns.tolist()

In [35]:
tnum_cols

['team',
 'targeted_productivity',
 'smv',
 'wip',
 'over_time',
 'incentive',
 'idle_time',
 'idle_men',
 'no_of_style_change',
 'no_of_workers']

In [36]:
tcat_cols

['quarter', 'department', 'day']

In [ ]:
Xs_train.isna().sum()

,0
quarter,0
department,0
day,0
team,0
targeted_productivity,0
smv,0
wip,415
over_time,0
incentive,0
idle_time,0


In [37]:
Xs_train[tnum_cols].median()

,0
team,6.00
targeted_productivity,0.75
smv,14.89
wip,1023.00
over_time,3960.00
incentive,0.00
idle_time,0.00
idle_men,0.00
no_of_style_change,0.00
no_of_workers,34.00


In [ ]:
wip_median = Xs_train['wip'].median()

Xs_train['wip'] = Xs_train['wip'].fillna(wip_median)
Xs_test['wip'] = Xs_test['wip'].fillna(wip_median)

In [38]:
Xs_train.isna().sum()

,0
quarter,0
department,0
day,0
team,0
targeted_productivity,0
smv,0
wip,0
over_time,0
incentive,0
idle_time,0


In [39]:
mean= Xs_train[tnum_cols].mean()
sd= Xs_train[tnum_cols].std(ddof=0).replace(0, 1.0)
train= (Xs_train[tnum_cols]-mean)/sd
test= (Xs_test[tnum_cols]-mean)/sd

Xs_train[tnum_cols]= train
Xs_test[tnum_cols]= test

In [41]:
Xs_train[tnum_cols].mean()

,0
team,8.909627e-17
targeted_productivity,3.146212e-16
smv,-5.568517e-17
wip,2.598641e-17
over_time,-1.457095e-16
incentive,-1.392129e-17
idle_time,-1.484938e-17
idle_men,-1.856172e-18
no_of_style_change,4.083579e-17
no_of_workers,9.280861e-17


In [44]:
Xs_train[tcat_cols].value_counts()

quarter   department  day      
Quarter1  sweing      Wednesday    33
Quarter2  sweing      Tuesday      31
Quarter1  sweing      Tuesday      30
                      Thursday     30
Quarter2  sweing      Monday       29
Quarter1  sweing      Saturday     29
Quarter2  sweing      Sunday       29
Quarter1  sweing      Monday       27
Quarter2  sweing      Wednesday    26
          finishing   Sunday       24
                      Monday       24
Quarter1  sweing      Sunday       24
Quarter2  finishing   Tuesday      23
                      Wednesday    22
Quarter1  finishing   Sunday       22
                      Monday       22
                      Wednesday    21
Quarter3  sweing      Sunday       20
Quarter1  finishing   Tuesday      20
                      Saturday     20
Quarter4  sweing      Tuesday      20
Quarter2  sweing      Thursday     19
Quarter4  sweing      Saturday     19
                      Thursday     19
                      Wednesday    19
Quarter3  sweing      Tuesday      18
                      Wednesday    18
Quarter1  finishing   Thursday     18
Quarter4  finishing   Thursday     18
          sweing      Monday       17
Quarter2  sweing      Saturday     17
Quarter4  finishing   Saturday     16
Quarter3  sweing      Thursday     16
Quarter2  finishing   Thursday     15
                      Saturday     14
Quarter3  sweing      Monday       14
Quarter4  finishing   Monday       14
                      Wednesday    13
Quarter3  finishing   Wednesday    13
Quarter4  sweing      Sunday       13
Quarter3  finishing   Monday       13
                      Sunday       13
Quarter4  finishing   Tuesday      12
Quarter3  finishing   Thursday     12
Quarter4  finishing   Sunday       12
Quarter5  sweing      Thursday     10
Quarter3  finishing   Tuesday      10
Quarter5  finishing   Saturday      9
                      Thursday      8
Quarter3  sweing      Saturday      8
          finishing   Saturday      7
Quarter5  sweing      Saturday      7
Name: count, dtype: int64

In [45]:
for col in tcat_cols:
    uniq = Xs_train[col].unique()
    cats_to_encode = uniq[1:]

    for cat in cats_to_encode:
        col_name = f"{col}_{cat}"
        Xs_train[col_name] = (Xs_train[col] == cat).astype(float)
        Xs_test[col_name] = (Xs_test[col] == cat).astype(float)

    Xs_train.drop(columns=[col], inplace=True)
    Xs_test.drop(columns=[col], inplace=True)

In [49]:
Xs_train_final = Xs_train.values
Xs_test_final = Xs_test.values

In [55]:
class mlr:
  def fit(self, X, y):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    self.theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

  def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.theta


In [68]:
import time
t0 = time.perf_counter()
lin_reg = mlr()
lin_reg.fit(Xs_train_final, ys_reg)
slr_time = time.perf_counter() - t0

t0 = time.perf_counter()
ys_pred_reg = lin_reg.predict(Xs_test_final)
slr_pred_time = time.perf_counter() - t0

print("Learned intercept (beta_0):", lin_reg.theta[0])
print("Learned feature weights:", lin_reg.theta[1:])

Learned intercept (beta_0): 0.7014025550729834
Learned feature weights: [-0.03057258  0.07123262 -0.08382359  0.00599736 -0.01557949  0.00701309
  0.0025334  -0.02877838 -0.01739576  0.11104908  0.00245922 -0.0039341
 -0.00679967  0.10338714  0.04340777  0.00096188 -0.009448    0.00955134
  0.03142442  0.02856977]


In [79]:
smae = np.mean(np.abs(ys_test_reg - ys_pred_reg))
srmse = np.sqrt(np.mean((ys_test_reg - ys_pred_reg) ** 2))

ss_res = np.sum((ys_test_reg - ys_pred_reg) ** 2)
ss_tot = np.sum((ys_test_reg - np.mean(ys_test_reg)) ** 2)
sr2 = 1 - (ss_res / ss_tot)

print(f"MAE:  {smae:.4f}")
print(f"RMSE: {srmse:.4f}")
print(f"R²:   {sr2:.4f}")
print(f"Time taken:   {slr_time}")

MAE:  0.1088
RMSE: 0.1481
R²:   0.1736
Time taken:   0.0049477949996799


In [75]:
class slogr:
    def __init__(self, lr=0.01, epochs=2000):
        self.lr = lr
        self.epochs = epochs

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0

        for _ in range(self.epochs):
            z = X @ self.w + self.b
            y_hat = self._sigmoid(z)

            dw = (1 / m) * (X.T @ (y_hat - y))
            db = (1 / m) * np.sum(y_hat - y)

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, t=0.5):
        return (self.predict_proba(X) >= t).astype(int)

In [76]:
import time

t0= time.perf_counter()
slog_reg = slogr(lr=0.1, epochs=2000)
slog_reg.fit(Xs_train_final, ys_cls)
slogr_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_cls = slog_reg.predict(Xs_test_final)
slogr_pred_time = time.perf_counter() - t0

tp = np.sum((ys_test_cls == 1) & (y_pred_cls == 1))
tn = np.sum((ys_test_cls == 0) & (y_pred_cls == 0))
fp = np.sum((ys_test_cls == 0) & (y_pred_cls == 1))
fn = np.sum((ys_test_cls == 1) & (y_pred_cls == 0))

saccuracy = (tp + tn) / len(ys_test_cls)
sprecision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
srecall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
sf1 = 2 * (sprecision * srecall) / (sprecision + srecall) if (sprecision + srecall) > 0 else 0.0

print(f"Accuracy:  {saccuracy:.4f}")
print(f"Precision: {sprecision:.4f}")
print(f"Recall:    {srecall:.4f}")
print(f"F1 Score:  {sf1:.4f}")
print(f"Time taken:   {slogr_time}")

Accuracy:  0.7542
Precision: 0.7658
Recall:    0.9605
F1 Score:  0.8521
Time taken:   0.7138847019996319


#**Part C - Comparison and Optimization**

In [80]:
model_comparison_data = {
    'Model': [
        'Scikit-learn Linear Regression',
        'Custom Linear Regression',
        'Scikit-learn Logistic Regression',
        'Custom Logistic Regression'
    ],
    'MAE': [
        mae_sk, smae, np.nan, np.nan
    ],
    'RMSE': [
        rmse_sk, srmse, np.nan, np.nan
    ],
    'R2 Score': [
        r2_sk, sr2, np.nan, np.nan
    ],
    'Accuracy': [
        np.nan, np.nan, acc_sk, saccuracy
    ],
    'Precision': [
        np.nan, np.nan, prec_sk, sprecision
    ],
    'Recall': [
        np.nan, np.nan, rec_sk, srecall
    ],
    'F1 Score': [
        np.nan, np.nan, f1_sk, sf1
    ],
    'Training Time (s)': [
        lr_time, slr_time, logr_time, slogr_time
    ],
    'Prediction Time (s)': [
        lr_pred_time, slr_pred_time, logr_pred_time, slogr_pred_time
    ]
}

comparison_df = pd.DataFrame(model_comparison_data)

comparison_df


,Model,MAE,RMSE,R2 Score,Accuracy,Precision,Recall,F1 Score,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.108771,0.148134,0.173576,NaN,NaN,NaN,NaN,0.024441,0.000701
1,Custom Linear Regression,0.108771,0.148134,0.173576,NaN,NaN,NaN,NaN,0.004948,0.000461
2,Scikit-learn Logistic Regression,NaN,NaN,NaN,0.741667,0.762557,0.943503,0.843434,0.016331,0.000649
3,Custom Logistic Regression,NaN,NaN,NaN,0.754167,0.765766,0.960452,0.852130,0.713885,0.000166


Performance and Runtime Analysis

1. Linear Regression
- Metrics Match: Custom and Scikit-learn results are identical (MAE: 0.108771, RMSE: 0.148134, R2: 0.173576) because both solve the exact closed-form Normal Equation w = (X^T X)^(-1) X^T y.
- Runtime: Custom model fits faster (~0.0049s vs ~0.0244s) by performing direct NumPy matrix operations, avoiding Scikit-learn's internal validation overhead.

2. Logistic Regression
- Metrics Variance: Custom accuracy is slightly higher (75.42% vs 74.17%) because Scikit-learn applies L2 regularization by default (penalty='l2') to shrink weights, whereas the custom class uses unregularized Gradient Descent.
- Runtime: Scikit-learn fits significantly faster (~0.016s vs ~0.714s) because its C-compiled L-BFGS solver converges in ~30 iterations, compared to 2,000 Python for loop passes in the custom model.

3. Prediction Speed
- Inference: Both models predict in under 0.001s because inference only requires a single vectorized matrix product (X * w + b).